# Part 11 (Demo): Backtesting a New Factor



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
from statsmodels.api import OLS

import warnings
warnings.filterwarnings("ignore")

from simple_factor_process import FactorProcessing_CorrelationBetweenTwoFactors as FactorIC # 因子IC
from simple_factor_process import FactorTool_GetStockSplit as FactorGroup # 分组
from simple_factor_process import FactorPurify # 中性化

from 一级二级行业字典 import dict_I,dict_II


# stock data
sw_ind = pd.read_pickle('数据/IndexComponent_SWN_I.txt')
stock_close = pd.read_pickle('数据/StockQuote_ClosePrice_BackwardAdj.txt')
stock_open = pd.read_pickle('数据/StockQuote_OpenPrice_BackwardAdj.txt')
monthly_trading_day = pd.read_pickle('数据/monthly_trading_day.pkl')
monthly_trading_day['start_date'] = pd.to_datetime(monthly_trading_day['start_date'], format='%Y%m%d')
monthly_trading_day['end_date'] = pd.to_datetime(monthly_trading_day['end_date'], format='%Y%m%d')
start_date = pd.to_datetime('20120101', format='%Y%m%d')
end_date = pd.to_datetime('20231231', format='%Y%m%d')
filtered_trading_days = monthly_trading_day.loc[(monthly_trading_day['end_date'] >= start_date) & (monthly_trading_day['end_date'] <= end_date)]
stock_close.index = pd.to_datetime(stock_close.index)
stock_open.index = pd.to_datetime(stock_open.index)
stock_ret_monthly = stock_close.reindex(index = filtered_trading_days.end_date).pct_change() 
stock_ret_monthly_nextopen = stock_open.shift(-1).reindex(filtered_trading_days.end_date).pct_change()

# index data
index_relative_price = pd.read_pickle('IndexQuote_SWS_ClosePrice.txt')
index_relative_price_1 = index_relative_price[list(dict_I.keys())]
index_relative_price_2 = index_relative_price[list(dict_II.keys())]

index_relative_price.index = pd.to_datetime(index_relative_price.index)
index_relative_ret_monthly = index_relative_price.reindex(index = filtered_trading_days.end_date).pct_change()
index_relative_ret_monthly_nextopen = index_relative_price.shift(-1).reindex(filtered_trading_days.end_date).pct_change()

index_relative_price_1.index = pd.to_datetime(index_relative_price_1.index)
index_relative_ret_monthly_1 = index_relative_price_1.reindex(index = filtered_trading_days.end_date).pct_change()
index_relative_ret_monthly_nextopen_1 = index_relative_price_1.shift(-1).reindex(filtered_trading_days.end_date).pct_change()

index_relative_price_2.index = pd.to_datetime(index_relative_price_2.index)
index_relative_ret_monthly_2 = index_relative_price_2.reindex(index = filtered_trading_days.end_date).pct_change()
index_relative_ret_monthly_nextopen_2 = index_relative_price_2.shift(-1).reindex(filtered_trading_days.end_date).pct_change()

index_daily_return = index_relative_price.pct_change()
index_daily_return = index_daily_return.loc['20120101':'20231231', :]
index_daily_return.index = pd.to_datetime(index_daily_return.index)

index_daily_return_1 = index_relative_price_1.pct_change()
index_daily_return_1 = index_daily_return_1.loc['20120101':'20231231', :]
index_daily_return_1.index = pd.to_datetime(index_daily_return_1.index)

index_daily_return_2 = index_relative_price_2.pct_change()
index_daily_return_2 = index_daily_return_2.loc['20120101':'20231231', :]
index_daily_return_2.index = pd.to_datetime(index_daily_return_2.index)


def simple_factor_test(factor, use_data = 'this_close'):
    if use_data=='this_close':
        this_ret_data = index_relative_ret_monthly_2.shift(-1)
    else: 
        this_ret_data = index_relative_ret_monthly_nextopen_2.shift(-1)
    ic,rankic = FactorIC(factor,this_ret_data) # 计算因子的ic,rankic序列
    factor_group = FactorGroup(factor)
    condata = pd.concat([factor_group.unstack(),this_ret_data.unstack()],axis=1).dropna().reset_index()
    condata.columns =['stockcode','date','group_id','ret']
    group_ret = condata.groupby(['date','group_id'])['ret'].mean().unstack()
    return ic,rankic,group_ret,factor_group

# simple_factor_test(FACTOR)


In [9]:
ZT = (stock_close - stock_open ) / stock_open

# 构建因子 
# Asset Pricing
# Behavial Finance
# Macro



In [13]:
ZT.T.count()

1991-07-03       0
1991-07-04       0
1991-07-05       0
1991-07-08       0
1991-07-09       0
              ... 
2023-12-25    5330
2023-12-26    5331
2023-12-27    5332
2023-12-28    5334
2023-12-29    5335
Length: 7701, dtype: int64

In [7]:
ZT = (stock_close - stock_open ) / stock_open
number_of_stock = ZT.T.count()
ZT  = abs(ZT) >= 0.09
number_of_stock_toomuch = ZT.T.sum()
factor = number_of_stock_toomuch / number_of_stock
factor.replace(np.nan, 0, inplace = True)
factor

1991-07-03    0.000000
1991-07-04    0.000000
1991-07-05    0.000000
1991-07-08    0.000000
1991-07-09    0.000000
                ...   
2023-12-25    0.013508
2023-12-26    0.010692
2023-12-27    0.008815
2023-12-28    0.018748
2023-12-29    0.015370
Length: 7701, dtype: float64